# NB_Source_SharePoint_DFO_PDF

**Layer:** Source / Bronze ingestion
**Source:** SharePoint Online – document library folder `DFO` (PDF files)
**Target:** Lakehouse `Files/landing/sharepoint/dfo/...` (raw PDFs) + Delta table `bronze_dfo_pdf_pages` (one row per PDF page)

| Step | What it does |
|---|---|
| 1. Parameters | Overridable from a Data Pipeline (`Notebook` activity → base parameters) |
| 2. Config | Imports, constants, logging, run metadata |
| 3. Auth | Service principal → Microsoft Graph token (secrets from Azure Key Vault) |
| 4. Discover | Lists all PDFs under the `DFO` folder (optionally recursive) |
| 5. Incremental filter | Skips files whose SharePoint `eTag` is already loaded |
| 6. Land raw | Downloads PDFs to the Lakehouse `Files` area, partitioned by load date / batch |
| 7. Extract | Distributed page-level text + table extraction with `pdfplumber` (`mapInPandas`) |
| 8. Load | Writes to Delta (`replaceWhere` per file → re-runs are idempotent) |
| 9. Audit & exit | Appends a row to the audit table and returns a JSON summary to the pipeline |

**Prerequisites**
- A default Lakehouse attached to this notebook.
- Entra ID app registration with Microsoft Graph **application** permission `Sites.Selected` (granted *read* on the site) or `Sites.Read.All`.
- Tenant ID / client ID / client secret stored in Azure Key Vault; the user / workspace identity running the notebook needs *Get* on secrets.
- `pdfplumber` available – either in a Fabric **Environment** (recommended for pipeline runs) or via the `%pip` cell below.

## 1. Parameters

In [ ]:
# ---- SharePoint source ----
sp_hostname        = "contoso.sharepoint.com"     # tenant SharePoint host
sp_site_path       = "sites/Finance"              # server-relative site path
sp_library_name    = "Documents"                  # drive name ("Shared Documents" library shows as "Documents")
sp_folder_path     = "DFO"                        # folder inside the library
include_subfolders = True

# ---- Key Vault (service principal secrets) ----
key_vault_uri           = "https://<your-kv>.vault.azure.net/"
kv_tenant_id_secret     = "sp-tenant-id"
kv_client_id_secret     = "sp-client-id"
kv_client_secret_secret = "sp-client-secret"

# ---- Lakehouse target ----
landing_root  = "Files/landing/sharepoint/dfo"    # relative to the default Lakehouse
target_table  = "bronze_dfo_pdf_pages"
audit_table   = "audit_ingestion_log"

# ---- Run control ----
load_type = "incremental"                         # incremental | full
batch_id  = ""                                    # pass @pipeline().RunId from the pipeline

## 2. Libraries
Skip this cell if `pdfplumber` is already in the attached Environment.
Inline `%pip` is disabled for pipeline runs unless the Notebook activity passes `_inlineInstallationEnabled = True`.

In [ ]:
%pip install pdfplumber==0.11.4 -q

## 3. Config & helpers

In [ ]:
import io
import json
import logging
import os
import re
import time
import uuid
from datetime import datetime, timezone

import pandas as pd
import requests
from pyspark.sql import functions as F
from pyspark.sql import types as T

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
log = logging.getLogger("NB_Source_SharePoint_DFO_PDF")

NOTEBOOK_NAME = "NB_Source_SharePoint_DFO_PDF"
SOURCE_SYSTEM = "SharePoint"
SOURCE_OBJECT = f"{sp_hostname}/{sp_site_path}/{sp_library_name}/{sp_folder_path}"
GRAPH_URL     = "https://graph.microsoft.com/v1.0"

run_start = datetime.now(timezone.utc)
batch_id  = batch_id or str(uuid.uuid4())
load_type = load_type.strip().lower()
assert load_type in ("incremental", "full"), f"Invalid load_type: {load_type}"

batch_rel_path   = f"{landing_root}/{run_start:%Y/%m/%d}/{batch_id}"   # used by Spark
batch_local_path = f"/lakehouse/default/{batch_rel_path}"             # used by Python file I/O

log.info(f"Batch {batch_id} | load_type={load_type} | source={SOURCE_OBJECT}")

## 4. Authentication – Microsoft Graph

In [ ]:
def get_graph_token() -> str:
    """Client-credentials token for Microsoft Graph using secrets held in Key Vault."""
    tenant_id     = notebookutils.credentials.getSecret(key_vault_uri, kv_tenant_id_secret)
    client_id     = notebookutils.credentials.getSecret(key_vault_uri, kv_client_id_secret)
    client_secret = notebookutils.credentials.getSecret(key_vault_uri, kv_client_secret_secret)
    resp = requests.post(
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
            "scope": "https://graph.microsoft.com/.default",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


session = requests.Session()
session.headers.update({"Authorization": f"Bearer {get_graph_token()}"})


def graph_get(url: str, stream: bool = False, max_retries: int = 5) -> requests.Response:
    """GET with retry on throttling (429) and transient 5xx, honouring Retry-After."""
    if not url.startswith("http"):
        url = f"{GRAPH_URL}{url}"
    refreshed = False
    for attempt in range(1, max_retries + 1):
        resp = session.get(url, stream=stream, timeout=120)
        if resp.status_code == 401 and not refreshed and attempt < max_retries:
            # Token expires after ~1h; long first loads outlive it, so refresh once and retry
            session.headers["Authorization"] = f"Bearer {get_graph_token()}"
            refreshed = True
            continue
        if resp.status_code in (429, 500, 502, 503, 504) and attempt < max_retries:
            wait = int(resp.headers.get("Retry-After", 2 ** attempt))
            log.warning(f"Graph {resp.status_code} – retry {attempt}/{max_retries} in {wait}s")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp

## 5. Discover PDFs in the DFO folder

In [ ]:
def get_site_id() -> str:
    return graph_get(f"/sites/{sp_hostname}:/{sp_site_path}").json()["id"]


def get_drive_id(site_id: str) -> str:
    drives = graph_get(f"/sites/{site_id}/drives").json()["value"]
    for d in drives:
        if d["name"].lower() == sp_library_name.lower():
            return d["id"]
    raise ValueError(f"Library '{sp_library_name}' not found. Available: {[d['name'] for d in drives]}")


def list_children(url: str):
    """Yield every driveItem in a folder, following @odata.nextLink paging."""
    while url:
        page = graph_get(url).json()
        yield from page.get("value", [])
        url = page.get("@odata.nextLink")


def list_pdfs(drive_id: str) -> list:
    pdfs = []
    folders = [f"/drives/{drive_id}/root:/{sp_folder_path.strip('/')}:/children?$top=999"]
    while folders:
        for item in list_children(folders.pop()):
            if "folder" in item and include_subfolders:
                folders.append(f"/drives/{drive_id}/items/{item['id']}/children?$top=999")
            elif "file" in item and item["name"].lower().endswith(".pdf"):
                pdfs.append({
                    "source_file_id":       item["id"],
                    "source_file_name":     item["name"],
                    "source_folder_path":   item.get("parentReference", {}).get("path", "").split("root:", 1)[-1],
                    "source_web_url":       item.get("webUrl"),
                    "source_etag":          item.get("eTag"),
                    "source_size_bytes":    item.get("size"),
                    "source_created_at":    item.get("createdDateTime"),
                    "source_modified_at":   item.get("lastModifiedDateTime"),
                    "source_modified_by":   item.get("lastModifiedBy", {}).get("user", {}).get("displayName"),
                })
    return pdfs


site_id  = get_site_id()
drive_id = get_drive_id(site_id)
all_pdfs = list_pdfs(drive_id)
log.info(f"Found {len(all_pdfs)} PDF(s) in {SOURCE_OBJECT}")

## 6. Incremental filter

In [ ]:
table_exists = spark.catalog.tableExists(target_table)

if load_type == "incremental" and table_exists:
    loaded = {
        (r.source_file_id, r.source_etag)
        for r in spark.table(target_table)
        .where("extract_status <> 'failed'")   # failed files are retried next run
        .select("source_file_id", "source_etag").distinct().collect()
    }
    files_to_load = [f for f in all_pdfs if (f["source_file_id"], f["source_etag"]) not in loaded]
else:
    files_to_load = all_pdfs

log.info(f"{len(files_to_load)} new/changed PDF(s) to ingest ({len(all_pdfs) - len(files_to_load)} unchanged)")

## 7. Land raw PDFs in the Lakehouse

In [ ]:
def safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]", "_", name)


def download_pdf(drive_id: str, f: dict) -> str:
    """Stream the file to the batch landing folder; returns the landed file name."""
    landed = f"{f['source_file_id']}__{safe_name(f['source_file_name'])}"
    resp = graph_get(f"/drives/{drive_id}/items/{f['source_file_id']}/content", stream=True)
    with open(f"{batch_local_path}/{landed}", "wb") as fh:
        for chunk in resp.iter_content(chunk_size=4 * 1024 * 1024):
            fh.write(chunk)
    return landed


download_errors = []
if files_to_load:
    os.makedirs(batch_local_path, exist_ok=True)
    for f in files_to_load:
        try:
            f["landed_file_name"] = download_pdf(drive_id, f)
        except Exception as e:
            log.error(f"Download failed for {f['source_file_name']}: {e}")
            download_errors.append({"file": f["source_file_name"], "error": str(e)[:500]})

landed_files = [f for f in files_to_load if f.get("landed_file_name")]
log.info(f"Landed {len(landed_files)} file(s) to {batch_rel_path}")

## 8. Extract text & tables (distributed)
One output row per page. Pages with no extractable text are flagged `needs_ocr`
(likely scanned images – route those to Azure AI Document Intelligence if needed).

In [ ]:
page_schema = T.StructType([
    T.StructField("landed_file_name", T.StringType()),
    T.StructField("page_number",      T.IntegerType()),
    T.StructField("page_count",       T.IntegerType()),
    T.StructField("page_text",        T.StringType()),
    T.StructField("tables_json",      T.StringType()),   # list of tables, each a list of rows
    T.StructField("table_count",      T.IntegerType()),
    T.StructField("extract_status",   T.StringType()),   # success | needs_ocr | failed
    T.StructField("error_message",    T.StringType()),
])
PAGE_COLS = [f.name for f in page_schema.fields]


def extract_pdf_pages(batches):
    import pdfplumber
    for pdf in batches:
        rows = []
        for r in pdf.itertuples(index=False):
            try:
                with pdfplumber.open(io.BytesIO(r.content)) as doc:
                    n_pages = len(doc.pages)
                    for i, page in enumerate(doc.pages, start=1):
                        text   = page.extract_text() or ""
                        tables = page.extract_tables() or []
                        rows.append((r.landed_file_name, i, n_pages, text, json.dumps(tables), len(tables),
                                     "success" if text.strip() else "needs_ocr", None))
            except Exception as e:
                rows.append((r.landed_file_name, None, None, None, None, 0, "failed", str(e)[:1000]))
        yield pd.DataFrame(rows, columns=PAGE_COLS)


if landed_files:
    raw_df = (
        spark.read.format("binaryFile")
        .load(batch_rel_path)
        .withColumn("landed_file_name", F.element_at(F.split("path", "/"), -1))
        .select("landed_file_name", "content")
    )
    n_parts = max(1, min(len(landed_files), spark.sparkContext.defaultParallelism))
    pages_df = raw_df.repartition(n_parts).mapInPandas(extract_pdf_pages, schema=page_schema)

    manifest_df = spark.createDataFrame(pd.DataFrame(landed_files).astype(str).replace({"None": None}))
    bronze_df = (
        pages_df.join(manifest_df, "landed_file_name", "inner")
        .withColumn("source_size_bytes",  F.col("source_size_bytes").cast("long"))
        .withColumn("source_created_at",  F.to_timestamp("source_created_at"))
        .withColumn("source_modified_at", F.to_timestamp("source_modified_at"))
        .withColumn("landed_file_path",   F.concat_ws("/", F.lit(batch_rel_path), "landed_file_name"))
        .withColumn("_source_system",     F.lit(SOURCE_SYSTEM))
        .withColumn("_source_object",     F.lit(SOURCE_OBJECT))
        .withColumn("_batch_id",          F.lit(batch_id))
        .withColumn("_ingested_at",       F.current_timestamp())
        .select(
            "source_file_id", "source_file_name", "source_folder_path", "source_web_url", "source_etag",
            "source_size_bytes", "source_created_at", "source_modified_at", "source_modified_by",
            "landed_file_path", "page_number", "page_count", "page_text", "tables_json", "table_count",
            "extract_status", "error_message",
            "_source_system", "_source_object", "_batch_id", "_ingested_at",
        )
        .cache()
    )
    display(bronze_df.groupBy("source_file_name", "extract_status").count().orderBy("source_file_name"))

## 9. Load to Delta
- **full**: overwrite the table.
- **incremental**: `replaceWhere` on the file IDs in this batch, so a changed PDF fully replaces its old pages (no orphan pages if the page count shrank).

In [ ]:
pages_written, files_failed = 0, len(download_errors)

if landed_files:
    writer = bronze_df.write.format("delta").option("mergeSchema", "true")
    if not table_exists:
        writer.mode("overwrite").saveAsTable(target_table)
    elif load_type == "full":
        writer.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)
    else:
        ids = ", ".join(f"'{f['source_file_id']}'" for f in landed_files)
        writer.mode("overwrite").option("replaceWhere", f"source_file_id IN ({ids})").saveAsTable(target_table)

    stats = bronze_df.agg(
        F.count("*").alias("pages"),
        F.countDistinct(F.when(F.col("extract_status") == "failed", F.col("source_file_id"))).alias("failed"),
    ).first()
    pages_written = stats.pages
    files_failed += stats.failed
    bronze_df.unpersist()

log.info(f"Wrote {pages_written} page row(s) to {target_table}; {files_failed} file(s) failed")

## 10. Audit log & exit

In [ ]:
run_status = "Succeeded" if files_failed == 0 else "SucceededWithErrors"
summary = {
    "notebook":        NOTEBOOK_NAME,
    "batch_id":        batch_id,
    "load_type":       load_type,
    "source_object":   SOURCE_OBJECT,
    "target_table":    target_table,
    "files_found":     len(all_pdfs),
    "files_processed": len(landed_files),
    "files_failed":    files_failed,
    "pages_written":   int(pages_written),
    "status":          run_status,
    "errors":          json.dumps(download_errors)[:4000],
    "run_start":       run_start.isoformat(),
    "run_end":         datetime.now(timezone.utc).isoformat(),
}

audit_df = (
    spark.createDataFrame(pd.DataFrame([summary]))
    .withColumn("run_start", F.to_timestamp("run_start"))
    .withColumn("run_end",   F.to_timestamp("run_end"))
)
audit_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(audit_table)

log.info(json.dumps(summary, indent=2))
notebookutils.notebook.exit(json.dumps(summary))